In [ ]:
# Переустанавливаем numpy и gensim, чтобы они были совместимы
!pip install --upgrade --force-reinstall numpy gensim
# После этой команды необходимо перезапустить рантайм (Runtime → Restart runtime)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.2
    Uninstalling wrapt-1.17.2:
      Successfully uninstalled wrapt-1.17.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: smart-open
    Found existing installation: s

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import CountVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from google.colab import drive

# Настройка matplotlib для кириллицы
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

def read_full_texts(folder_path, prefix='train_'):
    """
    Считывает полные тексты из .txt файлов префикса prefix.
    Возвращает списки текстов и авторов.
    """
    texts, labels = [], []
    for fn in os.listdir(folder_path):
        if fn.startswith(prefix) and fn.endswith('.txt'):
            author = fn[len(prefix):].replace('.txt','')
            path = os.path.join(folder_path, fn)
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    texts.append(f.read())
            except UnicodeDecodeError:
                with open(path, 'r', encoding='cp1251') as f:
                    texts.append(f.read())
            labels.append(author)
    return texts, labels

def split_into_fixed_fragments(texts, labels, fragment_len=1500, min_len=1000):
    """
    Делит каждый текст на фрагменты длиной fragment_len, счёт идёт с начала до конца.
    Оставляет только те, что >= min_len.
    Возвращает списки фрагментов и их авторов.
    """
    frags, frag_labels = [], []
    for txt, lab in zip(texts, labels):
        for i in range(0, len(txt), fragment_len):
            frag = txt[i:i+fragment_len]
            if len(frag) >= min_len:
                frags.append(frag)
                frag_labels.append(lab)
    return frags, frag_labels

def plot_metrics(history):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Точность на обучении')
    plt.title('График точности'); plt.xlabel('Эпоха'); plt.ylabel('Точность'); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Потери на обучении')
    plt.title('График потерь'); plt.xlabel('Эпоха'); plt.ylabel('Потери'); plt.legend()

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=labels, yticklabels=labels, cmap='Blues')
    plt.xlabel('Предсказано'); plt.ylabel('Истинное значение')
    plt.title('Матрица ошибок')
    plt.show()

def print_key_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    df = pd.DataFrame({
        'Метрика': [
            'Точность',
            'Precision (macro)',
            'Recall (macro)',
            'F1-score (macro)'
        ],
        'Значение': [
            f"{acc:.2f}",
            f"{precision:.2f}",
            f"{recall:.2f}",
            f"{f1:.2f}"
        ]
    })
    display(df)

# Сохранение модели и её компонентов
def save_model(model, le, vectorizer, model_path, le_path, vec_path):
    model.save(model_path)  # Сохранение модели
    np.save(le_path, le.classes_)  # Сохранение классов для лейблов
    with open(vec_path, 'wb') as f:
        pickle.dump(vectorizer, f)  # Сохранение vectorizer

if __name__ == '__main__':
    drive.mount('/content/drive')

    base = '/content/drive/MyDrive/Colab Notebooks/Lab_6/Выборка'
    # 1) Считываем 20 полнотекстовых файлов
    train_texts_full, train_labels_full = read_full_texts(os.path.join(base, 'Train'), 'train_')
    test_texts_full,  test_labels_full  = read_full_texts(os.path.join(base, 'Test'),  'test_')

    # 2) Разбиваем на фрагменты фиксированной длины
    fragment_len = 1500
    min_len      = 1000

    train_texts, train_labels = split_into_fixed_fragments(
        train_texts_full, train_labels_full,
        fragment_len=fragment_len,
        min_len=min_len
    )
    test_texts, test_labels = split_into_fixed_fragments(
        test_texts_full, test_labels_full,
        fragment_len=fragment_len,
        min_len=min_len
    )

    # 3) Параметры словаря
    max_words = 5000
    print(f"Используем max_words={max_words}")
    print(f"Фрагментов обучения: {len(train_texts)}, фрагментов теста: {len(test_texts)}")

    # 4) Преобразование текста в BoW
    vectorizer = CountVectorizer(max_features=max_words)
    X_train = vectorizer.fit_transform(train_texts).toarray()
    X_test  = vectorizer.transform(test_texts).toarray()

    # 5) Кодирование меток
    le      = LabelEncoder()
    y_train = to_categorical(le.fit_transform(train_labels))
    y_test  = to_categorical(le.transform(test_labels))

    # 6) Модель Dense для BoW
    model = Sequential([
        Dense(64, input_dim=max_words, activation='relu'),
        Dense(32, activation='relu'),
        Dense(y_train.shape[1], activation='softmax')
    ])
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    # 7) Обучение
    history = model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=32,
        validation_split=0.1
    )

    # 8) Предсказания и оценка
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)

    print_key_metrics(y_true, y_pred)
    plot_metrics(history)
    plot_confusion_matrix(y_true, y_pred, le.classes_)

    # 9) Сохранение модели
    # Сохранение обученной модели и её компонентов
    save_model(model, le, vectorizer,
        '/content/drive/MyDrive/Colab Notebooks/Lab_6/Trained_Model/Bag_of_words/my_model.h5',
        '/content/drive/MyDrive/Colab Notebooks/Lab_6/Trained_Model/Bag_of_words/classes.npy',
        '/content/drive/MyDrive/Colab Notebooks/Lab_6/Trained_Model/Bag_of_words/vectorizer.pkl')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

In [4]:
import os
import numpy as np
import pickle
from google.colab import drive
from tensorflow.keras.models import load_model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder

# Преобразование текста в формат BoW
def predict_style(model, vectorizer, text, le):
    X_input = vectorizer.transform([text]).toarray()
    probs = model.predict(X_input)[0]  # Получаем вероятности (предсказание для одного текста)

    top_idx = np.argmax(probs)
    predicted_label = le.inverse_transform([top_idx])[0]
    confidence = probs[top_idx]

    return predicted_label, confidence


# Загрузка модели и компонентов
def load_saved_model(model_path, le_path, vec_path):
    model = load_model(model_path)
    le_classes = np.load(le_path)
    le = LabelEncoder()
    le.classes_ = le_classes
    with open(vec_path, 'rb') as f:
        vectorizer = pickle.load(f)
    return model, le, vectorizer

# Основной запуск
if __name__ == '__main__':
    # Загрузка обученного
    drive.mount('/content/drive')

    model_path = '/content/drive/MyDrive/Colab Notebooks/Lab_6/Trained_Model/Bag_of_words/my_model.h5'
    le_path = '/content/drive/MyDrive/Colab Notebooks/Lab_6/Trained_Model/Bag_of_words/classes.npy'
    vec_path = '/content/drive/MyDrive/Colab Notebooks/Lab_6/Trained_Model/Bag_of_words/vectorizer.pkl'

    model, le, vectorizer = load_saved_model(model_path, le_path, vec_path)

    # Твоя авторская проверка
    file_path = '/content/drive/MyDrive/Colab Notebooks/Lab_6/Выборка/test_My.txt'
    with open(file_path, 'r', encoding='utf-8') as f:
        custom_text = f.read()

    result, confidence = predict_style(model, vectorizer, custom_text, le)
    print(f'Предсказанный стиль текста: {result} с уверенностью: {confidence:.2%})')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
🤖 Предсказанный стиль твоего текста: Каверин (уверенность: 94.58%)
